## Final Model

In [ ]:
import os
import re
from tqdm import tqdm
import pandas as pd
from collections import Counter

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Mounted at /content/drive
Using device: cuda


In [3]:
# configuration.py

LANG_CODES = ["de", "en", "es", "fr", "it", "ko", "pt", "ta", "be", "ru"]
MAX_SENTENCES_PER_LANG = 50000
TEST_RATIO = 0.2
RANDOM_SEED = 42

NLP_PATH= "/content/drive/MyDrive/NLP"

CONLLU_DATA_DIR = f"{NLP_PATH}/preprocessing/data"
TWITTER_CSV_PATH = f"{NLP_PATH}/final_solution/data/tweets_dataset.csv"
STOPWORDS_PATH = f"{NLP_PATH}/final_solution/data/stopwords"


In [12]:
# load_data.py

def clean_text_basic(text):
    text = re.sub(r"https?://\S+", " ", text)

    cleaned_chars = []
    for ch in text:
        if ch.isalpha() or ch.isspace():
            cleaned_chars.append(ch)
        else:
            cleaned_chars.append(" ")

    text = "".join(cleaned_chars)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def load_conllu_sentences(path):
    """Load sentences from a CoNLL-U file."""
    sentences = []
    current_tokens = []
    current_text = None

    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")

                if line.startswith("# text = "):
                    current_text = line.split("=", 1)[1].strip()
                elif not line:
                    if current_text:
                        cleaned = clean_text_basic(current_text)
                        if cleaned and len(cleaned) > 5:
                            sentences.append(cleaned)
                    elif current_tokens:
                        sent = " ".join(current_tokens)
                        cleaned = clean_text_basic(sent)
                        if cleaned and len(cleaned) > 5:
                            sentences.append(cleaned)
                    current_tokens = []
                    current_text = None
                elif line.startswith("#"):
                    continue
                else:
                    cols = line.split("\t")
                    if len(cols) >= 2:
                        current_tokens.append(cols[1])
    except FileNotFoundError:
        print(f"File not found: {path}")
        return []

    return sentences

def load_wikipedia_data(max_per_lang=MAX_SENTENCES_PER_LANG):
    """Load Wikipedia data."""
    texts = []
    labels = []

    print("Loading Wikipedia...")
    for lang in tqdm(LANG_CODES):
        path = os.path.join(CONLLU_DATA_DIR, f"output_{lang}.conllu")
        sents = load_conllu_sentences(path)

        if max_per_lang and len(sents) > max_per_lang:
            sents = sents[:max_per_lang]

        texts.extend(sents)
        labels.extend([lang] * len(sents))
        print(f"  {lang}: {len(sents)} sentences")

    print(f"Total: {len(texts)} sentences")
    return texts, labels

def load_twitter_data():
    """Load Twitter data."""
    print("Loading Twitter...")

    if not os.path.exists(TWITTER_CSV_PATH):
        print(f"File not found: {TWITTER_CSV_PATH}")
        return [], []

    df = pd.read_csv(TWITTER_CSV_PATH)

    TWITTER_LANG_MAPPING = {
        "en": "en", "de": "de", "es": "es", "fr": "fr",
        "it": "it", "ko": "ko", "pt": "pt", "ru": "ru",
        "be": "be", "ta": "ta"
    }

    texts = []
    labels = []

    for _, row in df.iterrows():
        text = str(row.get("content", ""))
        lang = str(row.get("language", ""))

        if lang in TWITTER_LANG_MAPPING:
            cleaned = clean_text_basic(text)
            if cleaned and len(cleaned) > 5:
                texts.append(cleaned)
                labels.append(TWITTER_LANG_MAPPING[lang])

    print(f"Total: {len(texts)} tweets")
    print(f"Distribution: {Counter(labels)}")
    return texts, labels


In [5]:
# rulebased.py

from dataclasses import dataclass
import re
import unicodedata
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Iterable, Set, Optional

def _load_stopwords(lang: str, base_dir: str = STOPWORDS_PATH) -> Set[str]:
    file_path = os.path.join(base_dir, f"{lang}.txt")

    if not os.path.exists(file_path):
        return set()

    out: Set[str] = set()
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            out.add(s.lower())
    return out

def _build_lang_rules(langs: Iterable[str], base_dir: str = STOPWORDS_PATH) -> Dict[str, Dict]:
    meta: Dict[str, Dict] = {
        "en": {"script": "latin", "special_chars": set()},
        "de": {"script": "latin", "special_chars": {"ä", "ö", "ü", "ß"}},
        "es": {"script": "latin", "special_chars": {"ñ"}},
        "fr": {"script": "latin", "special_chars": {"é", "è", "ê", "à", "ç", "ù", "ô"}},
        "it": {"script": "latin", "special_chars": {"à", "è", "é", "ì", "ò", "ù"}},
        "pt": {"script": "latin", "special_chars": {"ã", "õ"}},
        "ko": {"script": "hangul", "special_chars": set()},
        "ta": {"script": "tamil", "special_chars": set()},
        "ru": {"script": "cyrillic", "special_chars": {"ъ", "ы", "э"}},
        "be": {"script": "cyrillic", "special_chars": {"ў", "і"}},
    }

    rules: Dict[str, Dict] = {}
    for lang in langs:
        m = meta.get(lang, {"script": "mixed", "special_chars": set()})
        rules[lang] = {
            "script": m["script"],
            "stopwords": _load_stopwords(lang, base_dir=base_dir),
            "special_chars": m["special_chars"],
        }
    return rules


@dataclass
class RuleTaggerConfig:
    max_sent_per_lang: int = 50000
    top_k_n_grams: int = 40
    weights: str = "base"
    stopwords_dir: Optional[str] = None


class RuleTagger:
    WORD_RE = re.compile(r"[^\W\d_]+", re.UNICODE)

    WEIGHT_PRESETS = {
        "base": dict(stopword=3.0, special=4.0, bi=1.0, tri=2.0, four=3.0)
    }

    SCRIPT_TOKEN = {
        "latin": "<SCRIPT=LATIN>",
        "cyrillic": "<SCRIPT=CYRILLIC>",
        "hangul": "<SCRIPT=HANGUL>",
        "tamil": "<SCRIPT=TAMIL>",
        "mixed": "<SCRIPT=MIXED>",
    }

    LANG_RULES: Dict[str, Dict] = {}

    def __init__(self, cfg: RuleTaggerConfig):
        self.cfg = cfg
        self.w = self.WEIGHT_PRESETS[cfg.weights]

        stop_dir = cfg.stopwords_dir or STOPWORDS_PATH
        self.LANG_RULES = _build_lang_rules(LANG_CODES, base_dir=stop_dir)

        self.ngrams = {lang: {"bi": set(), "tri": set(), "four": set()} for lang in LANG_CODES}
        self.fitted = False

    # ---------- core utils ----------
    def detect_script(self, text: str) -> str:
        has_cyr = has_lat = has_hangul = has_tamil = False
        for ch in text:
            if not ch.isalpha():
                continue
            name = unicodedata.name(ch, "")
            if "CYRILLIC" in name:
                has_cyr = True
            elif "LATIN" in name:
                has_lat = True
            elif "HANGUL" in name:
                has_hangul = True
            elif "TAMIL" in name:
                has_tamil = True

        flags = [has_cyr, has_lat, has_hangul, has_tamil]
        if sum(flags) == 1:
            if has_cyr: return "cyrillic"
            if has_lat: return "latin"
            if has_hangul: return "hangul"
            if has_tamil: return "tamil"
        return "mixed"

    def tokenize(self, text: str) -> List[str]:
        return [w.lower() for w in self.WORD_RE.findall(text)]

    def char_ngrams(self, text: str, n: int) -> Counter:
        text = text.lower()
        chars = [c for c in text if c.isalpha()]
        grams = ["".join(chars[i:i+n]) for i in range(len(chars) - n + 1)]
        return Counter(grams)

    # ---------- training n-grams ----------
    def _build_top_ngrams(
        self,
        texts: Iterable[str],
        labels: Iterable[str],
        n: int,
        top_k: int,
        min_freq: int = 5,
    ) -> Dict[str, List[str]]:
        per_lang = defaultdict(Counter)
        for t, y in zip(texts, labels):
            per_lang[y].update(self.char_ngrams(t, n))
        out = {}
        for lang, c in per_lang.items():
            c = Counter({g: cnt for g, cnt in c.items() if cnt >= min_freq})
            out[lang] = [g for g, _ in c.most_common(top_k)]
        return out

    def fit(self, train_texts: List[str], train_labels: List[str]) -> None:
        k = self.cfg.top_k_n_grams
        bi = self._build_top_ngrams(train_texts, train_labels, 2, k)
        tri = self._build_top_ngrams(train_texts, train_labels, 3, k)
        four = self._build_top_ngrams(train_texts, train_labels, 4, k)

        for lang in LANG_CODES:
            if lang in bi: self.ngrams[lang]["bi"] = set(bi[lang])
            if lang in tri: self.ngrams[lang]["tri"] = set(tri[lang])
            if lang in four: self.ngrams[lang]["four"] = set(four[lang])

        self.fitted = True

    # ---------- scoring + prediction ----------
    def score_language(self, text: str, lang: str) -> float:
        rules = self.LANG_RULES[lang]
        toks = self.tokenize(text)

        bi_cnt = self.char_ngrams(text, 2)
        tri_cnt = self.char_ngrams(text, 3)
        four_cnt = self.char_ngrams(text, 4)

        s = 0.0

        for tok in toks:
            if tok in rules["stopwords"]:
                s += self.w["stopword"]

        for ch in text.lower():
            if ch in rules["special_chars"]:
                s += self.w["special"]

        for g in self.ngrams[lang]["bi"]:
            s += self.w["bi"] * bi_cnt.get(g, 0)
        for g in self.ngrams[lang]["tri"]:
            s += self.w["tri"] * tri_cnt.get(g, 0)
        for g in self.ngrams[lang]["four"]:
            s += self.w["four"] * four_cnt.get(g, 0)

        return s

    def rule_predict(self, text: str) -> Tuple[str, float, float]:
        script = self.detect_script(text)
        candidates = [
            lang for lang in LANG_CODES
            if script == "mixed" or self.LANG_RULES[lang]["script"] == script
        ] or LANG_CODES[:]

        scores = {lang: self.score_language(text, lang) for lang in candidates}
        best_lang, best_score = max(scores.items(), key=lambda x: x[1])
        sorted_scores = sorted(scores.values(), reverse=True)
        second = sorted_scores[1] if len(sorted_scores) > 1 else 0.0
        margin = best_score - second
        return best_lang, best_score, margin

    def conf_bin(self, best_score: float, margin: float) -> str:
        if best_score <= 0:
            return "LOW"
        if margin >= 15:
            return "HIGH"
        if margin >= 5:
            return "MID"
        return "LOW"

    def prefix(self, text: str) -> str:
        script_tok = self.SCRIPT_TOKEN.get(self.detect_script(text), "<SCRIPT=MIXED>")

        if not self.fitted:
            return f"{script_tok} <RB=en> <RB_CONF=LOW>"

        rb_lang, best_score, margin = self.rule_predict(text)
        conf = self.conf_bin(best_score, margin)
        return f"{script_tok} <RB={rb_lang}> <RB_CONF={conf}>"

    def special_tokens(self) -> List[str]:
        script_tokens = list(self.SCRIPT_TOKEN.values())
        rb_lang_tokens = [f"<RB={l}>" for l in LANG_CODES]
        rb_conf_tokens = ["<RB_CONF=LOW>", "<RB_CONF=MID>", "<RB_CONF=HIGH>"]
        return script_tokens + rb_lang_tokens + rb_conf_tokens

In [6]:
# finetuned_xlmroberta.py

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm


class FineTunedRobertaModel:
    def __init__(self, tagger: RuleTagger):
        self.tagger = tagger

        self.model_name = "xlm-roberta-base"
        self.max_length = 128
        self.batch_size = 32
        self.num_epochs = 3
        self.learning_rate = 2e-5

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using: {self.device}")

        self.label2id = {lang: i for i, lang in enumerate(LANG_CODES)}
        self.id2label = {i: lang for i, lang in enumerate(LANG_CODES)}

        self.model = None
        self.tokenizer = None

    def _load_model(self):
        print(f"Loading {self.model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)

        # add tagger's tokens
        self.tokenizer.add_special_tokens({
            "additional_special_tokens": self.tagger.special_tokens()
        })

        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=len(LANG_CODES),
            id2label=self.id2label,
            label2id=self.label2id,
        )

        self.model.resize_token_embeddings(len(self.tokenizer))
        self.model.to(self.device)
        print("Model loaded")

    def _apply_prefixes(self, texts):
        return [f"{self.tagger.prefix(t)} {t}" for t in texts]

    def train(self, train_texts, train_labels, max_samples=50_000):
        if self.model is None:
            self._load_model()

        # cap samples like before
        if len(train_texts) > max_samples:
            idx = np.random.choice(len(train_texts), max_samples, replace=False)
            train_texts = [train_texts[i] for i in idx]
            train_labels = [train_labels[i] for i in idx]

        # IMPORTANT: fit rule tagger on the training split
        self.tagger.fit(train_texts, train_labels)

        tagged_texts = self._apply_prefixes(train_texts)

        enc = self.tokenizer(
            tagged_texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        y = torch.tensor([self.label2id[l] for l in train_labels], dtype=torch.long)

        dataset = TensorDataset(enc["input_ids"], enc["attention_mask"], y)
        dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        optimizer = AdamW(self.model.parameters(), lr=self.learning_rate)
        self.model.train()

        for epoch in range(self.num_epochs):
            total_loss, correct, total = 0.0, 0, 0
            pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{self.num_epochs}")
            for input_ids, attention_mask, yb in pbar:
                input_ids = input_ids.to(self.device)
                attention_mask = attention_mask.to(self.device)
                yb = yb.to(self.device)

                optimizer.zero_grad()
                out = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=yb)
                loss = out.loss
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
                preds = torch.argmax(out.logits, dim=-1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)

                pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{correct/total:.4f}"})

            print(f"Epoch {epoch+1}: loss={total_loss/len(dataloader):.4f}, acc={correct/total:.4f}")

        print("Completed")

    def predict(self, texts):
        if self.model is None or self.tokenizer is None:
            self._load_model()

        self.model.eval()
        tagged_texts = self._apply_prefixes(texts)

        preds_all = []
        for i in tqdm(range(0, len(tagged_texts), self.batch_size), desc="Predict"):
            batch = tagged_texts[i:i+self.batch_size]
            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt"
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            with torch.no_grad():
                out = self.model(**inputs)
                preds = torch.argmax(out.logits, dim=-1).cpu().tolist()

            preds_all.extend([self.id2label[p] for p in preds])

        return preds_all

In [7]:
# main.py

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


def main():
    # Load Dataset
    wiki_texts, wiki_labels = load_wikipedia_data(max_per_lang=50000)

    # Split train/test
    wiki_train_texts, wiki_test_texts, wiki_train_labels, wiki_test_labels = train_test_split(
        wiki_texts, wiki_labels,
        test_size=TEST_RATIO,
        random_state=RANDOM_SEED,
        stratify=wiki_labels
    )

    print(f"\nWikipedia Train: {len(wiki_train_texts)}")
    print(f"Wikipedia Test: {len(wiki_test_texts)}")

    twitter_texts, twitter_labels = load_twitter_data()

    # Train Fine-Tuned XLM-RoBERTa Model
    tagger = RuleTagger(RuleTaggerConfig(top_k_n_grams=40, max_sent_per_lang=50000, weights="base"))
    finetune_roberta_model = FineTunedRobertaModel(tagger)
    finetune_roberta_model.train(wiki_train_texts, wiki_train_labels, max_samples=50000)

    # Evaluation on Wikipedia
    print("\n" + "="*50)
    print("[EVALUATION ON WIKIPEDIA]")
    print("="*50)

    wiki_preds = finetune_roberta_model.predict(wiki_test_texts)
    wiki_acc = accuracy_score(wiki_test_labels, wiki_preds)
    print(f"\nAccuracy: {wiki_acc:.4f} ({wiki_acc*100:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(wiki_test_labels, wiki_preds))

    # Evaluation on Twitter
    if twitter_texts:
        print("\n" + "="*50)
        print("[EVALUATION ON TWITTER]")
        print("="*50)

        twitter_preds = finetune_roberta_model.predict(twitter_texts)
        twitter_acc = accuracy_score(twitter_labels, twitter_preds)
        print(f"\nAccuracy: {twitter_acc:.4f} ({twitter_acc*100:.2f}%)")
        print("\nClassification Report:")
        print(classification_report(twitter_labels, twitter_preds))

    # Final summary
    print("\n" + "="*50)
    print("FINAL SUMMARY")
    print("="*50)
    print(f"XLM-RoBERTa on Wikipedia: {wiki_acc*100:.2f}%")
    if twitter_texts:
        print(f"XLM-RoBERTa on Twitter: {twitter_acc*100:.2f}%")
        print(f"Drop: {(wiki_acc - twitter_acc)*100:.2f}%")


if __name__ == "__main__":
    main()


Loading Wikipedia...


 10%|█         | 1/10 [00:53<08:02, 53.64s/it]

  de: 50000 sentences


 20%|██        | 2/10 [01:35<06:12, 46.53s/it]

  en: 50000 sentences


...

Epoch 3: loss=0.0461, acc=0.9861
Completed

[EVALUATION ON WIKIPEDIA]


Predict: 100%|██████████| 3125/3125 [09:17<00:00,  5.60it/s]



Accuracy: 0.9856 (98.56%)

Classification Report:
              precision    recall  f1-score   support

          be       0.99      0.99      0.99     10000
          de       0.98      0.98      0.98     10000
          en       0.98      0.98      0.98     10000
          es       0.96      0.98      0.97     10000
          fr       0.98      0.99      0.99     10000
          it       1.00      0.98      0.99     10000
          ko       1.00      1.00      1.00     10000
          pt       0.98      0.97      0.98     10000
          ru       0.99      0.99      0.99     10000
          ta       1.00      1.00      1.00     10000

    accuracy                           0.99    100000
   macro avg       0.99      0.99      0.99    100000
weighted avg       0.99      0.99      0.99    100000


[EVALUATION ON TWITTER]


Predict: 100%|██████████| 75/75 [00:08<00:00,  8.75it/s]



Accuracy: 0.8737 (87.37%)

Classification Report:
              precision    recall  f1-score   support

          de       0.95      0.88      0.91       300
          en       0.75      0.90      0.81       297
          es       0.67      0.79      0.72       294
          fr       0.90      0.74      0.81       293
          it       0.97      0.86      0.91       298
          ko       1.00      1.00      1.00       299
          pt       0.83      0.82      0.83       295
          ru       1.00      1.00      1.00       300

    accuracy                           0.87      2376
   macro avg       0.88      0.87      0.88      2376
weighted avg       0.88      0.87      0.88      2376


FINAL SUMMARY
XLM-RoBERTa on Wikipedia: 98.56%
XLM-RoBERTa on Twitter: 87.37%
Drop: 11.18%


## Baseline Model

### Rule Based

In [13]:
import re
import unicodedata
import random
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Iterable

# ---------------------------
# Final Chosen Configuration
# max_sent_per_lang = 50,000
# top_k_n_grams = 40
# weights = base
#   stopwords: 3.0
#   language-specific char: 4.0
#   bigram match: 1.0
#   trigram match: 2.0
#   four-gram match: 3.0

LANG_CODES = ["de", "en", "es", "fr", "it", "ko", "pt", "ta", "be", "ru"]

LANG_RULES: Dict[str, Dict] = {
    "en": {
        "name": "English",
        "script": "latin",
        "stopwords": set(),
        "special_chars": set(),
        "bigrams": set(),
        "trigrams": set(),
        "fourgrams": set(),
    },
    "de": {
        "name": "German",
        "script": "latin",
        "stopwords": set(),
        "special_chars": {"ä", "ö", "ü", "ß"},
        "bigrams": set(),
        "trigrams": set(),
        "fourgrams": set(),
    },
    "es": {
        "name": "Spanish",
        "script": "latin",
        "stopwords": set(),
        "special_chars": {"ñ"},
        "bigrams": set(),
        "trigrams": set(),
        "fourgrams": set(),
    },
    "fr": {
        "name": "French",
        "script": "latin",
        "stopwords": set(),
        "special_chars": {"é", "è", "ê", "à", "ç", "ù", "ô"},
        "bigrams": set(),
        "trigrams": set(),
        "fourgrams": set(),
    },
    "it": {
        "name": "Italian",
        "script": "latin",
        "stopwords": set(),
        "special_chars": {"à", "è", "é", "ì", "ò", "ù"},
        "bigrams": set(),
        "trigrams": set(),
        "fourgrams": set(),
    },
    "pt": {
        "name": "Portuguese",
        "script": "latin",
        "stopwords": set(),
        "special_chars": {"ã", "õ"},
        "bigrams": set(),
        "trigrams": set(),
        "fourgrams": set(),
    },
    "ko": {
        "name": "Korean",
        "script": "hangul",
        "stopwords": set(),
        "special_chars": set(),
        "bigrams": set(),
        "trigrams": set(),
        "fourgrams": set(),
    },
    "ta": {
        "name": "Tamil",
        "script": "tamil",
        "stopwords": set(),
        "special_chars": set(),
        "bigrams": set(),
        "trigrams": set(),
        "fourgrams": set(),
    },
    "ru": {
        "name": "Russian",
        "script": "cyrillic",
        "stopwords": set(),
        "special_chars": {"ъ", "ы", "э"},
        "bigrams": set(),
        "trigrams": set(),
        "fourgrams": set(),
    },
    "be": {
        "name": "Belarusian",
        "script": "cyrillic",
        "stopwords": set(),
        "special_chars": {"ў", "і"},
        "bigrams": set(),
        "trigrams": set(),
        "fourgrams": set(),
    },
}

# ----------------------------
# Script detection

def detect_script(text: str) -> str:
    has_cyr = has_lat = has_hangul = has_tamil = False

    for ch in text:
        if not ch.isalpha():
            continue
        name = unicodedata.name(ch, "")
        if "CYRILLIC" in name:
            has_cyr = True
        elif "LATIN" in name:
            has_lat = True
        elif "HANGUL" in name:
            has_hangul = True
        elif "TAMIL" in name:
            has_tamil = True

    flags = [has_cyr, has_lat, has_hangul, has_tamil]
    if sum(flags) == 1:
        if has_cyr:
            return "cyrillic"
        if has_lat:
            return "latin"
        if has_hangul:
            return "hangul"
        if has_tamil:
            return "tamil"
    return "mixed"

# -------------------------
# Tokenization + n-grams

WORD_RE = re.compile(r"[^\W\d_]+", re.UNICODE)

def tokenize(text: str) -> List[str]:
    return [w.lower() for w in WORD_RE.findall(text)]

def char_ngrams(text: str, n: int) -> Counter:
    text = text.lower()
    chars = [c for c in text if c.isalpha()]
    grams = ["".join(chars[i:i + n]) for i in range(len(chars) - n + 1)]
    return Counter(grams)

# -----------------------
# Read CoNLL-U -> sentences

def load_conllu_sentences(path: str) -> List[str]:
    sentences: List[str] = []
    current_tokens: List[str] = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            if not line:
                if current_tokens:
                    sent = clean_text_basic(" ".join(current_tokens))
                    if sent:
                        sentences.append(sent)
                    current_tokens = []
                continue

            if line.startswith("#"):
                continue

            cols = line.split("\t")
            if len(cols) >= 2:
                current_tokens.append(cols[1])

    if current_tokens:
        sent = clean_text_basic(" ".join(current_tokens))
        if sent:
            sentences.append(sent)

    return sentences

def load_corpus_from_dir(data_dir: str, max_sent_per_lang: int = 50000) -> Tuple[List[str], List[str]]:
    texts: List[str] = []
    labels: List[str] = []

    for lang in LANG_CODES:
        path = f"{data_dir}/output_{lang}.conllu"
        try:
            sents = load_conllu_sentences(path)
        except FileNotFoundError:
            print(f"WARNING: file not found for language {lang}: {path}")
            continue

        sents = sents[:max_sent_per_lang]
        texts.extend(sents)
        labels.extend([lang] * len(sents))

    return texts, labels

# ---------------------------------
# Build n-gram rules (top_k=40)

def build_char_ngrams_from_corpus(
    texts: Iterable[str],
    labels: Iterable[str],
    n: int,
    top_k: int = 40,
    min_freq: int = 5,
) -> Dict[str, List[str]]:
    per_lang_counts: Dict[str, Counter] = defaultdict(Counter)

    for text, lang in zip(texts, labels):
        per_lang_counts[lang].update(char_ngrams(text, n))

    result: Dict[str, List[str]] = {}
    for lang, counter in per_lang_counts.items():
        filtered = Counter({g: c for g, c in counter.items() if c >= min_freq})
        result[lang] = [g for g, _ in filtered.most_common(top_k)]
    return result

def init_ngram_rules(
    texts: Iterable[str],
    labels: Iterable[str],
    bigram_top_k: int = 40,
    trigram_top_k: int = 40,
    fourgram_top_k: int = 40,
) -> None:
    bigrams = build_char_ngrams_from_corpus(texts, labels, n=2, top_k=bigram_top_k)
    trigrams = build_char_ngrams_from_corpus(texts, labels, n=3, top_k=trigram_top_k)
    fourgrams = build_char_ngrams_from_corpus(texts, labels, n=4, top_k=fourgram_top_k)

    for lang, rules in LANG_RULES.items():
        if lang in bigrams:
            rules["bigrams"] = set(bigrams[lang])
        if lang in trigrams:
            rules["trigrams"] = set(trigrams[lang])
        if lang in fourgrams:
            rules["fourgrams"] = set(fourgrams[lang])

# ---------------------------------
# Scoring (base weights)

W_STOP = 3.0
W_SPECIAL = 4.0
W_BIGRAM = 1.0
W_TRIGRAM = 2.0
W_FOURGRAM = 3.0

def score_language(text: str, lang_code: str) -> float:
    rules = LANG_RULES[lang_code]
    tokens = tokenize(text)

    bigram_counts = char_ngrams(text, 2)
    trigram_counts = char_ngrams(text, 3)
    fourgram_counts = char_ngrams(text, 4)

    score = 0.0

    # stopwords
    sw = rules["stopwords"]
    for tok in tokens:
        if tok in sw:
            score += W_STOP

    # language-specific characters
    specials = rules["special_chars"]
    for ch in text.lower():
        if ch in specials:
            score += W_SPECIAL

    # n-grams
    for gram in rules["bigrams"]:
        score += W_BIGRAM * bigram_counts.get(gram, 0)

    for gram in rules["trigrams"]:
        score += W_TRIGRAM * trigram_counts.get(gram, 0)

    for gram in rules["fourgrams"]:
        score += W_FOURGRAM * fourgram_counts.get(gram, 0)

    return score

# ------------------------------
# Prediction

def predict_language(text: str) -> str:
    script = detect_script(text)

    candidates: List[str] = []
    for code, cfg in LANG_RULES.items():
        if script == "mixed" or cfg["script"] == script:
            candidates.append(code)

    if not candidates:
        candidates = list(LANG_RULES.keys())

    if len(candidates) == 1:
        return candidates[0]

    scores = {code: score_language(text, code) for code in candidates}
    best_lang, best_score = max(scores.items(), key=lambda x: x[1])

    return "unknown" if best_score == 0 else best_lang

# --------------------------
# Evaluation

def evaluate(texts: List[str], labels: List[str]) -> Tuple[float, Counter]:
    correct = 0
    conf = Counter()

    for t, gold in zip(texts, labels):
        pred = predict_language(t)
        if pred == gold:
            correct += 1
        conf[(gold, pred)] += 1

    acc = correct / len(texts) if texts else 0.0
    return acc, conf

# ---------------------------------
# Split + run

def train_test_split(texts: List[str], labels: List[str], test_ratio: float = 0.2, seed: int = 42):
    idx = list(range(len(texts)))
    random.Random(seed).shuffle(idx)
    cut = int(len(idx) * (1 - test_ratio))

    train_idx = idx[:cut]
    test_idx = idx[cut:]

    train_texts = [texts[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    test_texts = [texts[i] for i in test_idx]
    test_labels = [labels[i] for i in test_idx]

    return train_texts, train_labels, test_texts, test_labels


if __name__ == "__main__":
    for lang, rules in LANG_RULES.items():
      rules["stopwords"] = _load_stopwords(lang)

    texts, labels = load_corpus_from_dir(CONLLU_DATA_DIR, max_sent_per_lang=50000)
    print(f"Total sentences: {len(texts)}")

    train_texts, train_labels, test_texts, test_labels = train_test_split(
        texts, labels, test_ratio=0.2, seed=42
    )

    init_ngram_rules(
        train_texts,
        train_labels,
        bigram_top_k=40,
        trigram_top_k=40,
        fourgram_top_k=40,
    )

    print("\n" + "="*50)
    print("[EVALUATION ON WIKIPEDIA]")
    print("="*50)

    wiki_rb_preds = [predict_language(t) for t in test_texts]
    wiki_rb_acc = accuracy_score(test_labels, wiki_rb_preds)

    print(f"\nRule-based Accuracy: {wiki_rb_acc:.4f} ({wiki_rb_acc*100:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(
        test_labels,
        wiki_rb_preds,
        labels=LANG_CODES,
        zero_division=0
    ))



    # ---- Twitter test (domain shift evaluation) ----
    tw_texts, tw_labels = load_twitter_data()

    print("\n" + "="*50)
    print("[RULE-BASED EVALUATION ON TWITTER]")
    print("="*50)

    tw_rb_preds = [predict_language(t) for t in tw_texts]
    tw_rb_acc = accuracy_score(tw_labels, tw_rb_preds)

    print(f"\nAccuracy: {tw_rb_acc:.4f} ({tw_rb_acc*100:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(
        tw_labels,
        tw_rb_preds,
        labels=sorted(set(tw_labels) | set(tw_rb_preds)),
        zero_division=0
    ))





Total sentences: 500000

[EVALUATION ON WIKIPEDIA]

Rule-based Accuracy: 0.9488 (94.88%)

Classification Report:
              precision    recall  f1-score   support

          de       0.96      0.94      0.95     10018
          en       0.86      0.97      0.92     10017
          es       0.94      0.89      0.91      9844
          fr       0.96      0.95      0.96      9958
          it       0.94      0.95      0.95     10181
          ko       1.00      1.00      1.00      9842
          pt       0.96      0.90      0.93     10066
          ta       1.00      1.00      1.00     10095
          be       0.99      0.89      0.94      9946
          ru       0.90      0.99      0.94     10033

   micro avg       0.95      0.95      0.95    100000
   macro avg       0.95      0.95      0.95    100000
weighted avg       0.95      0.95      0.95    100000

Loading Twitter...
Total: 2976 tweets
Distribution: Counter({'ta': 300, 'be': 300, 'ru': 300, 'de': 300, 'ko': 299, 'it': 298, '

### Machine Learning Based (Naive Bayes)

In [20]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score


# Data Loading and Parsing
def parse_conllu_data(file_path, language_code):
    """Parses a custom CoNLL-U file and extracts sentences and their labels."""
    data = []
    current_sentence = ""

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line.startswith('# text = '):
                    if current_sentence:
                        data.append({'text': current_sentence, 'label': language_code})
                    current_sentence = line.split('=', 1)[1].strip()
                elif not line and current_sentence:
                    data.append({'text': current_sentence, 'label': language_code})
                    current_sentence = ""
            if current_sentence:
                 data.append({'text': current_sentence, 'label': language_code})

    except FileNotFoundError:
        return pd.DataFrame()

    return pd.DataFrame(data).drop_duplicates()

data_dir = CONLLU_DATA_DIR
languages = ['be', 'de', 'en', 'es', 'fr', 'it', 'ko', 'pt', 'ru', 'ta']

all_data = []
N_SAMPLES = 500000
print("--- Parsing Data ---")

for lang_code in languages:
    filename = f'output_{lang_code}.conllu'
    file_path = os.path.join(data_dir, filename)

    df = parse_conllu_data(file_path, lang_code)

    if df.empty:
        print(f"Warning: {filename} is empty.")
        continue

    if len(df) > N_SAMPLES // len(languages):
        df = df.sample(
            n=N_SAMPLES // len(languages),
            random_state=42
        )

    print(f"Parsed file: {filename} with {len(df)} sentences.")
    all_data.append(df)

df_combined = pd.concat(all_data, ignore_index=True)


# Data Preparation
total_samples = len(df_combined)

if total_samples == 0:
    print("\nError: No data was loaded.")
else:
    print(f"\nTotal sentences loaded: {total_samples}")

    if total_samples >= N_SAMPLES:
        df_sampled = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)
    else:
        df_sampled = df_combined
        print(f"Warning: Only {total_samples} available, using all data instead of {N_SAMPLES}.")

    le = LabelEncoder()
    df_sampled['label_id'] = le.fit_transform(df_sampled['label'])
    label_map = dict(zip(le.classes_, le.transform(le.classes_)))
    print("Label Mapping:", label_map)

    X_train, X_test, y_train, y_test = train_test_split(
        df_sampled['text'],
        df_sampled['label_id'],
        test_size=0.2,
        random_state=42,
        stratify=df_sampled['label_id']
    )

    vectorizer = TfidfVectorizer(max_features=1000)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    print(f"\nTraining set size: {X_train_vec.shape}")
    print(f"Testing set size: {X_test_vec.shape}")

    # Model Training and Evaluation
    print("\nTraining the Multinomial Naive Bayes Model...")

    model = MultinomialNB()
    model.fit(X_train_vec, y_train)

    y_pred = model.predict(X_test_vec)

    accuracy = accuracy_score(y_test, y_pred)
    print("\n" + "="*50)
    print("[EVALUATION ON WIKIPEDIA]")
    print("="*50)
    print(f"\nModel Accuracy: {accuracy:.4f}")

    print("\nClassification Report (Language ID -> Language Code):")
    class_names = le.classes_
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))



    # Evaluation on Twitter dataset
    twitter_texts, twitter_labels = load_twitter_data()

    allowed = set(le.classes_)
    filtered = [(t, y) for t, y in zip(twitter_texts, twitter_labels) if y in allowed]
    twitter_texts = [t for t, _ in filtered]
    twitter_labels = [y for _, y in filtered]

    if twitter_texts:
        print("\n" + "="*50)
        print("[EVALUATION ON TWITTER]")
        print("="*50)

        twitter_y = le.transform(twitter_labels)          # LabelEncoder
        twitter_X = vectorizer.transform(twitter_texts)   # TF-IDF

        twitter_preds = model.predict(twitter_X)
        twitter_acc = accuracy_score(twitter_y, twitter_preds)
        print(f"\nAccuracy: {twitter_acc:.4f} ({twitter_acc*100:.2f}%)")
        print("\nClassification Report:")

        present_codes = sorted(set(twitter_labels))
        present_ids = le.transform(present_codes)

        print(classification_report(
            twitter_y,
            twitter_preds,
            labels=present_ids,
            target_names=present_codes,
            zero_division=0
        ))

--- Parsing Data ---
Parsed file: output_be.conllu with 50000 sentences.
Parsed file: output_de.conllu with 50000 sentences.
Parsed file: output_en.conllu with 50000 sentences.
Parsed file: output_es.conllu with 50000 sentences.
Parsed file: output_fr.conllu with 50000 sentences.
Parsed file: output_it.conllu with 50000 sentences.
Parsed file: output_ko.conllu with 50000 sentences.
Parsed file: output_pt.conllu with 50000 sentences.
Parsed file: output_ru.conllu with 50000 sentences.
Parsed file: output_ta.conllu with 50000 sentences.

Total sentences loaded: 500000
Label Mapping: {'be': np.int64(0), 'de': np.int64(1), 'en': np.int64(2), 'es': np.int64(3), 'fr': np.int64(4), 'it': np.int64(5), 'ko': np.int64(6), 'pt': np.int64(7), 'ru': np.int64(8), 'ta': np.int64(9)}

Training set size: (400000, 1000)
Testing set size: (100000, 1000)

Training the Multinomial Naive Bayes Model...

[EVALUATION ON WIKIPEDIA]

Model Accuracy: 0.8599

Classification Report (Language ID -> Language Code):
